# 04 · Reaction transition states

A TS has partial forming/breaking bonds an FF can't type — so rxembed pins the reacting core to its known geometry and conformer-searches only the periphery. The core comes from a **TS `.xyz`** (`freeze=`, grafted back to 0.000 Å) or from **SMILES** (`distances=` the *exact* reacting-atom values, not loose windows). The gate is made **TS-aware** by naming the reacting core in `frozen=`. In the renders, the genuine forming/breaking bonds are drawn with `ts_bonds=` (dashed) and labelled with `labels=["i j d"]` — everything else (ordinary bonds) is auto-detected. xyzrender is 1-indexed, so atoms are passed `+1`.

In [ ]:
import tempfile
import numpy as np
import rxembed as rx
import xyzrender
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolTransforms as T
from rxembed.embed.dispatch import _xyz_to_mol

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")


def xyz(ens):
    return ens.lowest(1).dump(tempfile.mktemp(suffix=".xyz"))  # lowest conformer -> path for xyzrender

## Frozen TS core from an `.xyz`

A real spirocyclization TS (the forming C–C is the reacting bond). Freeze the reacting atoms — the core is grafted back **exactly** on every pose while the rest is searched. The forming bond is drawn dashed with its held length; the TS-aware gate confirms a clean periphery.

In [ ]:
name = "spiro-ts1"
rc = [14, 15]  # reacting-core atoms (from a graphrc trajectory in practice)
bonds = [(14, 15)]  # forming / breaking bonds held during embed
ens = rx.embed(f"structures/{name}.xyz", freeze=rc, n=4).mc(preset="rapid").prune()
Pin = _xyz_to_mol(f"structures/{name}.xyz", 0).GetConformer().GetPositions()
drift = max(
    abs(T.GetBondLength(ens.mol.GetConformer(c), i, j) - float(np.linalg.norm(Pin[i] - Pin[j])))
    for c in ens.ids
    for i, j in bonds
)
print(f"{len(ens)} conformers | forming bond {tuple(bonds[0])} held to {drift:.4f} Å")
rep = rx.geometry.check(ens.mol, conf_id=ens.ids[0], frozen=rc, reference=f"structures/{name}.xyz")
print("geometry.check:", "CLEAN ✓" if rep.ok() else f"FAIL ✗ {rep.violations[:6]}")
xyzrender.render(
    xyz(ens), no_hy=True, ts_bonds=[(i + 1, j + 1) for i, j in bonds], labels=[f"{i + 1} {j + 1} d" for i, j in bonds]
)

## Constrained TS from SMILES — a 3-centre SN2 (exact distances)

No geometry needed: give the reacting-atom distances **exactly** (a single value is held to ±0.05 Å, tight — a TS is a point on the surface, not a loose window) and the near-linear angle. The forming F···C and breaking C···Cl are the TS bonds.

In [ ]:
core = {("[F-]", "[CH2][Cl]"): 2.02, ("[CH2][Cl]", "[Cl-,Cl]"): 2.28}  # exact forming/breaking lengths (Å)
sn2 = rx.embed("[F-].CCCCCl", distances=core, angles={("[F-]", "[CH2]Cl", "Cl"): 178}).mc(preset="rapid").prune()
print(
    f"{len(sn2)} SN2-guess conformers | F···C {sn2.measure(('[F-]', '[CH2][Cl]'))['mean']:.2f} Å, "
    f"C···Cl {sn2.measure(('[CH2][Cl]', '[Cl-,Cl]'))['mean']:.2f} Å, "
    f"F-C-Cl {sn2.measure(('[F-]', '[CH2]Cl', 'Cl'))['mean']:.0f}°"
)
f = sn2.mol.GetSubstructMatch(Chem.MolFromSmarts("[F-]"))[0]
c = sn2.mol.GetSubstructMatch(Chem.MolFromSmarts("[CH2][Cl]"))[0]
cl = sn2.mol.GetSubstructMatch(Chem.MolFromSmarts("[Cl-,Cl]"))[0]
xyzrender.render(
    xyz(sn2),
    no_hy=True,
    ts_bonds=[(f + 1, c + 1), (c + 1, cl + 1)],
    labels=[f"{f + 1} {c + 1} d", f"{c + 1} {cl + 1} d"],
)

## A real reaction — FLP borylation, and the TS-aware gate

A frustrated-Lewis-pair borylation TS from SMILES: five exact role-distances define the concerted B–H···C–H transfer. With `distances=` there is no `frozen` set, so the gate reads the forming B···C bond as a clash — name the reacting core in `frozen=` and it becomes TS-aware.

In [ ]:
def flp_ts_core(mol):
    """The FLP-borylation reacting atoms + their exact TS distances (a concerted B-H...C-H transfer)."""
    B = next(a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == "B")
    amineN = next(a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == "N" and not a.GetIsAromatic())
    pyrroleN = next(a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == "N" and a.GetIsAromatic())
    C = next(
        nb.GetIdx()
        for nb in mol.GetAtomWithIdx(pyrroleN).GetNeighbors()
        if nb.GetIsAromatic() and any(h.GetSymbol() == "H" for h in nb.GetNeighbors())
    )
    H = next(h.GetIdx() for h in mol.GetAtomWithIdx(C).GetNeighbors() if h.GetSymbol() == "H")
    core = {(B, H): 2.08, (amineN, H): 1.51, (C, H): 1.29, (B, C): 1.69, (B, amineN): 3.06}  # exact TS (Å)
    return core, [B, H, C, amineN, pyrroleN]


smi = "CB(C1=C(N(C)C)C=CC=C1).CN1C=CC=C1"  # aryl-borane + 1-methylpyrrole
core, react = flp_ts_core(Chem.AddHs(Chem.MolFromSmiles(smi)))
flp = rx.embed(smi, distances=core).mc(preset="rapid").prune()
B, H, C, aN, pN = react
print(f"{len(flp)} FLP-borylation TS conformers")
print("gate WITHOUT frozen (forming B···C looks like a clash):")
lo = flp.lowest(1)
rep = rx.geometry.check(lo.mol, conf_id=lo.ids[0])
print("geometry.check:", "CLEAN ✓" if rep.ok() else f"FAIL ✗ {rep.violations[:6]}")
print("gate WITH frozen=core (TS-aware):")
lo = flp.lowest(1)
rep = rx.geometry.check(lo.mol, conf_id=lo.ids[0], frozen=react)
print("geometry.check:", "CLEAN ✓" if rep.ok() else f"FAIL ✗ {rep.violations[:6]}")
ts = [(B, H), (aN, H), (C, H)]  # the three transferring-H bonds
xyzrender.render(
    xyz(flp), no_hy=True, ts_bonds=[(i + 1, j + 1) for i, j in ts], labels=[f"{i + 1} {j + 1} d" for i, j in ts]
)

## Capstone — a TS with *multiple* non-covalent grips

A bifunctional iminophosphorane–thiourea TS (172 atoms, 3 fragments). `freeze=` holds the reacting core exactly while rxembed **discovers the distinct NCI binding modes at the TS geometry** — the catalyst gripping the substrate several ways — and conf-searches each. The seeded contacts are drawn with `nci_bonds=` (dotted), distinct from the frozen reacting core.

In [ ]:
name = "bimp"
rc = [10, 11, 12, 14]  # reacting-core atoms (from a graphrc trajectory in practice)
modes = rx.nci_modes(_xyz_to_mol(f"structures/{name}.xyz", 0))  # binding modes AT the frozen TS geometry
print(f"{len(modes)} NCI binding modes discovered at the bimp TS; searching the first two (core frozen):")
best, shown = None, 0
for label, mode in modes.items():
    ens = rx.embed(f"structures/{name}.xyz", freeze=rc, contacts=mode, n=2).mc(preset="rapid").prune()
    if not ens.ids:
        continue
    shown += 1
    print(f"  grip {shown}: {len(ens):2d} confs | {label[:46]}")
    best = best or (ens, mode)
    if shown >= 2:
        break
ens, mode = best
grip = [(min(k) + 1, max(k) + 1) for k in mode.distances]  # the seeded H-bond contacts
xyzrender.render(xyz(ens), no_hy=True, nci_bonds=grip)